In [111]:
import numpy as py
import pandas as pd

Loading data

In [112]:
df = pd.read_csv("./messy-transactions.csv")
customers = pd.read_csv("./customers.csv")


In [102]:
def check(df, label, expected_rows=None):
    print(f"{label:<28} {len(df):>7,} rows")
    if expected_rows is not None:
        assert len(df) == expected_rows, f"expected {expected_rows}, got {len(df)}"
    return df

In [103]:
df.head(20)

,transaction_id,customer_id,order_date,amount,quantity,channel
0,T004716,C00313,2024-08-23T23:26:00Z,121.77,4,In-Store
1,T003183,C00329,26/09/2024,$145.95,2,In-Store
2,T002691,C00177,"June 26, 2024",$143.98,4,Web
3,T001589,C00723,"May 28, 2024",135.19,1,Mobile
4,T001242,C00798,"September 3, 2024",$100.10,2,In-Store
5,T005166,C00536,08/03/2024,$9.23,3,web
6,T004813,C00106,2024-11-04,1.00,1,Web
7,T001326,C00554,01/08/2024,$1.00,NaN,Mobile
8,T004910,C00857,2024-08-14,50.69,4,mobile
9,T002136,C00869,16/08/2024,155.90,NaN,Mobile


In [104]:
check(df, "Raw transactions").shape

Raw transactions               8,432 rows


(8432, 6)

In [71]:
df.shape

(8432, 6)

In [113]:
amount = df['amount']
amount.nunique()

7285

In [73]:
amount.info()

<class 'pandas.Series'>
RangeIndex: 8432 entries, 0 to 8431
Series name: amount
Non-Null Count  Dtype
--------------  -----
8432 non-null   str  
dtypes: str(1)
memory usage: 66.0 KB


In [74]:
customers


,customer_id,signup_date,segment,region
0,C00001,2022-08-17T05:45:00Z,smb,east
1,C00001,2022-08-17T05:45:00Z,smb,north
2,C00002,21/12/2022,smb,east
3,C00002,21/12/2022,smb,west
4,C00003,2022-02-03,enterprise,north
...,...,...,...,...
1015,C00896,2022-07-20T04:32:00Z,enterprise,east
1016,C00897,2022-04-09T18:21:00Z,smb,south
1017,C00898,2022-11-25,enterprise,south
1018,C00899,2022-04-17T09:52:00Z,enterprise,south


In [107]:
quantity = df['quantity']
quantity.sample(25)

811       5
1584      5
7558      5
7810      1
8313      2
8244      3
4423    NaN
1237      4
6398      3
6165      3
738       5
1536      5
5548      4
3920      5
2006      1
5795      4
4479      5
7424      5
1197      3
2307      5
7555      3
3757      1
2438      5
7351      2
4324      4
Name: quantity, dtype: str

Clean Exact duplicate rows

In [108]:
df.duplicated().sum()

np.int64(432)

In [109]:
df = df.drop_duplicates()

In [110]:
check(df, "Raw transactions").shape

Raw transactions               8,000 rows


(8000, 6)

Clean Money stored as strings, three different ways

In [78]:
print(df["amount"])
print(df["amount"].dtype)

0         121.77
1        $145.95
2        $143.98
3         135.19
4        $100.10
          ...   
8425       69.71
8427       93.06
8428      73.05 
8430     131.46 
8431       40.76
Name: amount, Length: 8000, dtype: str
str


In [79]:
df["amount"].dropna().unique()

<StringArray>
[  '121.77',  '$145.95',  '$143.98',   '135.19',  '$100.10',    '$9.23',
   ' 1.00 ',    '$1.00',    '50.69', ' 155.90 ',
 ...
   '110.60',   '132.05', ' 119.22 ',    '96.32', ' 109.83 ',   '$81.47',
    '93.06',  ' 73.05 ', ' 131.46 ',    '40.76']
Length: 7285, dtype: str

Clean the whitespace

In [114]:
clean = df["amount"].str.strip()

Remove the dollar sign

In [120]:
clean = clean.str.replace("$", "", regex=False)

AttributeError: Can only use .str accessor with string values, not floating

Convert to numerical

In [122]:
clean = pd.to_numeric(clean)

In [123]:
clean.sum()

np.float64(1012908.85)

In [83]:
print(clean.dtype)

float64


In [84]:
print(df["amount"].dtype)
print(df["amount"].head())

str
0     121.77
1    $145.95
2    $143.98
3     135.19
4    $100.10
Name: amount, dtype: str


In [85]:
df["amount"] = clean
print(df["amount"].dtype)
print(df["amount"].head())

float64
0    121.77
1    145.95
2    143.98
3    135.19
4    100.10
Name: amount, dtype: float64


Clean Leading and trailing whitespace in a join key

In [86]:
df["customer_id"].duplicated()

0       False
1       False
2       False
3       False
4       False
        ...  
8425     True
8427     True
8428     True
8430     True
8431     True
Name: customer_id, Length: 8000, dtype: bool

In [87]:
df["customer_id"].dropna().unique()

<StringArray>
[  'C00313',   'C00329',   'C00177',   'C00723',   'C00798',   'C00536',
   'C00106',   'C00554',   'C00857',   'C00869',
 ...
 ' C00518 ', ' C00392 ', ' C00628 ', ' C00266 ', ' C00475 ', ' C00768 ',
 ' C00175 ', ' C00045 ', ' C00846 ', ' C00065 ']
Length: 1445, dtype: str

In [88]:
df["customer_id"] = df["customer_id"].str.strip()

In [89]:
df["customer_id"].dropna().unique()

<StringArray>
['C00313', 'C00329', 'C00177', 'C00723', 'C00798', 'C00536', 'C00106',
 'C00554', 'C00857', 'C00869',
 ...
 'C00230', 'C00376', 'C00543', 'C00784', 'C00048', 'C00472', 'C00717',
 'C00204', 'C00379', 'C00216']
Length: 900, dtype: str

In [90]:
customers["customer_id"].dropna().unique()

<StringArray>
['C00001', 'C00002', 'C00003', 'C00004', 'C00005', 'C00006', 'C00007',
 'C00008', 'C00009', 'C00010',
 ...
 'C00891', 'C00892', 'C00893', 'C00894', 'C00895', 'C00896', 'C00897',
 'C00898', 'C00899', 'C00900']
Length: 900, dtype: str

In [91]:
customers["customer_id"] = customers["customer_id"].str.strip()

Clean Inconsistent casing and stray spaces

In [92]:
df["channel"] = df["channel"].str.strip()
df["channel"] = df["channel"].str.lower()



In [93]:
df["channel"].unique()

<StringArray>
['in-store', 'web', 'mobile']
Length: 3, dtype: str

Clean Four date formats in one column

In [94]:
df["order_date"].dropna().unique()

<StringArray>
['2024-08-23T23:26:00Z',           '26/09/2024',        'June 26, 2024',
         'May 28, 2024',    'September 3, 2024',           '08/03/2024',
           '2024-11-04',           '01/08/2024',           '2024-08-14',
           '16/08/2024',
 ...
 '2024-02-11T02:12:00Z', '2024-01-26T05:30:00Z', '2024-04-05T08:20:00Z',
 '2024-01-05T21:21:00Z', '2024-01-03T14:07:00Z', '2024-03-05T14:24:00Z',
 '2024-08-15T23:49:00Z', '2024-12-19T12:20:00Z', '2024-04-04T21:46:00Z',
 '2024-03-16T11:41:00Z']
Length: 3031, dtype: str

In [95]:
raw = df["order_date"]

In [96]:
df["order_date"] = pd.to_datetime(
    format="mixed",
    dayfirst=True,
    utc=True,
    errors="coerce"
)

TypeError: to_datetime() missing 1 required positional argument: 'arg'

In [ ]:
assert df["order_date"].isna().sum() == 0

In [ ]:
df["order_date"].isna().sum()

np.int64(0)

In [ ]:
def check(df, label, expected_rows=None):
    print(f"{label:<28} {len(df):>7,} rows")
    if expected_rows is not None:
        assert len(df) == expected_rows, f"expected {expected_rows}, got {len(df)}"
    return df



In [ ]:
df["quantity"].value_counts(dropna=False)

quantity
4      1552
3      1548
5      1545
1      1504
2      1457
NaN     320
         74
Name: count, dtype: int64

In [ ]:
q = df["quantity"]
q

0       4
1       2
2       4
3       1
4       2
       ..
8425    2
8427    1
8428    5
8430    4
8431    2
Name: quantity, Length: 8000, dtype: str

In [ ]:
q = df["quantity"].astype("string").str.strip().str.lower()

In [ ]:
missing_values = {"na", " ", "n/a", "null", ""}
q = q.replace(list(missing_values), pd.NA)

In [ ]:
df["quantity"] = q

In [ ]:
df["quantity"].isna().sum()
# df["quantity"].notna().sum()

np.int64(320)

In [ ]:
assert not df["quantity"].isin(
    ["na", " ", "n/a", "null", ""]
).any()

In [ ]:
df["quantity"].dropna()

0       4
1       2
2       4
3       1
4       2
       ..
8425    2
8427    1
8428    5
8430    4
8431    2
Name: quantity, Length: 7680, dtype: str

In [97]:
customers = customers.drop_duplicates("customer_id")

In [98]:
df = df.merge(customers, on="customer_id", how="left", validate="many_to_one")


In [99]:
df.shape

(8000, 9)

In [ ]:
customers["customer_id"].duplicated().sum()

np.int64(120)

In [ ]:
customers.loc[
    customers["customer_id"].duplicated(keep=False)
]

,customer_id,signup_date,segment,region
0,C00001,2022-08-17T05:45:00Z,smb,east
1,C00001,2022-08-17T05:45:00Z,smb,north
2,C00002,21/12/2022,smb,east
3,C00002,21/12/2022,smb,west
4,C00003,2022-02-03,enterprise,north
...,...,...,...,...
235,C00118,2022-09-03,smb,north
236,C00119,03/08/2022,consumer,east
237,C00119,03/08/2022,consumer,north
238,C00120,"May 24, 2022",enterprise,east


In [ ]:
dupes = customers[
    customers["customer_id"].duplicated(keep=False)
].sort_values("customer_id")

print(dupes)

    customer_id           signup_date     segment region
0        C00001  2022-08-17T05:45:00Z         smb   east
1        C00001  2022-08-17T05:45:00Z         smb  north
2        C00002            21/12/2022         smb   east
3        C00002            21/12/2022         smb   west
4        C00003            2022-02-03  enterprise  north
..          ...                   ...         ...    ...
235      C00118            2022-09-03         smb  north
236      C00119            03/08/2022    consumer   east
237      C00119            03/08/2022    consumer  north
238      C00120          May 24, 2022  enterprise   east
239      C00120          May 24, 2022  enterprise  south

[240 rows x 4 columns]


In [ ]:
dupes.groupby("customer_id").agg({
    "signup_date": "nunique",
    "segment": "nunique",
    "region": "nunique"
}).head()

,signup_date,segment,region
customer_id,,,
C00001,1,1,2
C00002,1,1,2
C00003,1,1,2
C00004,1,1,2
C00005,1,1,2


In [ ]:
df["amount"].isna().sum()

np.int64(0)

In [ ]:
df["amount"].sum()

np.float64(961769.21)

In [ ]:
df["customer_id"].nunique()

900

In [ ]:
# customers.shape 


AttributeError: 'DataFrame' object has no attribute 'unique'